In [1]:
import pandas as pd
import numpy as np
import io

In [ ]:
# ==== File Paths ====
duelist_file = r"Input_Files/duelist_dump_16_march.xlsx"
duelist_main_file = r"Z:/1.Reports Repository/FY 2082.83/1. Duelist/9.Chaitra/Duelist 15th March, 2026.xlsx"
insurance = r"Z:/1.Reports Repository/FY 2082.83/1. Duelist/9.Chaitra/Chaitra Insurance 2082.xlsx"
output_file = r"Output_Files/updated_duelist_march_16.xlsx"

insurance = pd.read_excel(insurance, dtype={"MainCode": str})

if duelist_file.endswith(".xlsb"):
    duelist = pd.read_excel(
        duelist_file,
        dtype={"MainCode": str, "AcCodeForChg": str, "Nominee": str},
        engine="pyxlsb"
)
else:
    duelist = pd.read_excel(
        duelist_file,
        dtype={"MainCode": str, "AcCodeForChg": str, "Nominee": str},
        engine="openpyxl"
    )  

if duelist_main_file.endswith(".xlsb"):
    duelist_main = pd.read_excel(
        duelist_main_file,
        dtype={"MainCode": str, "AcCodeForChg": str, "Nominee": str},
        engine="pyxlsb"
    )
else:
    duelist_main = pd.read_excel(
        duelist_main_file,
        dtype={"MainCode": str, "AcCodeForChg": str, "Nominee": str},
        engine="openpyxl"
    )

print(len(duelist))
duelist.head(2)

26822


,BBB,AT,AcTypeDesc,CC,OldClientCode,ClientCode,OldAcNum,MainCode,Name,Address,...,ModelDetails,LotNumber,DownPayment,VehicleModel,VehicleNo,Region,Obligor,DistrictCode,IntRateOfChgAc,ExpiryDateOfChgAc
0,001,20,HP-3WH (TEMPO),01,NaN,00024235,NaN,00102000024235000002,BISHONATH MAHATO,SAKHUWA PRASAUNI SAKHUWA PRASAUNI PARSA,...,FOTON OMEGA STREAM,NaN,40% Downpayment,5 FOTON,NaN,NaN,NaN,025,17.0,17/01/2026
1,001,20,HP-3WH (TEMPO),01,~~~~~,~~~~~,~~,~~,~~GROUP TOTAL 1,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [4]:
def clean_columns(df):
    df.columns = df.columns.str.strip()
    return df

duelist = clean_columns(duelist)
duelist_main = clean_columns(duelist_main)
insurance = clean_columns(insurance)

In [5]:
insurance.dtypes

BBB                          int64
AT                          object
AcTypeDesc                  object
ClientCode                   int64
MainCode                    object
Name                        object
InsPolicyNo                 object
InsIssueDate                object
InsMaturityDate             object
Date                datetime64[ns]
InsuredAmt                 float64
InsPremium                 float64
InsurancePremium           float64
Insurer                     object
MortgageCode                object
Identifier                  object
Phone                        int64
MobileNo                     int64
Mobile1                     object
dtype: object

In [6]:
insurance["Date"] = pd.to_datetime(insurance["Date"])

In [7]:
yesterday = pd.Timestamp.today().normalize() - pd.Timedelta(days=1)

insurance["InsurancePremium"] = np.where(
    yesterday < insurance["Date"],
    insurance["InsPremium"],
    0
)

In [8]:
duelist = duelist.copy()
duelist = duelist[(duelist['ClientCode']!="~~~~~")]
print(len(duelist))
duelist.head(2)

26767


,BBB,AT,AcTypeDesc,CC,OldClientCode,ClientCode,OldAcNum,MainCode,Name,Address,...,ModelDetails,LotNumber,DownPayment,VehicleModel,VehicleNo,Region,Obligor,DistrictCode,IntRateOfChgAc,ExpiryDateOfChgAc
0,001,20,HP-3WH (TEMPO),01,NaN,00024235,NaN,00102000024235000002,BISHONATH MAHATO,SAKHUWA PRASAUNI SAKHUWA PRASAUNI PARSA,...,FOTON OMEGA STREAM,NaN,40% Downpayment,5 FOTON,NaN,NaN,NaN,025,17.0,17/01/2026
2,001,21,HP-2WH NEPAL ARMY SCHEME,01,NaN,00010670,NaN,00102100010670000006,SAIRA SHAKYA,GUITOLE LALITPUR,...,Ray ZR Disc Hybrid 125 Street Rally OBII,NaN,30% Downpayment,2 SCOOTER,NaN,CENTRAL,NaN,029,15.0,21/07/2025


In [9]:
print(len(duelist))
print(duelist["MainCode"].nunique())
print(duelist["MainCode"].duplicated().sum())
print("\n")

print(len(duelist_main))
print(duelist_main["MainCode"].nunique())
print(duelist_main["MainCode"].duplicated().sum())
print("\n")

print(len(insurance))
print(insurance["MainCode"].nunique())
print(insurance["MainCode"].duplicated().sum())

26767
26767
0


26799
26799
0


863
830
33


In [10]:
insurance[insurance.duplicated("MainCode", keep=False)].head(5)

,BBB,AT,AcTypeDesc,ClientCode,MainCode,Name,InsPolicyNo,InsIssueDate,InsMaturityDate,Date,InsuredAmt,InsPremium,InsurancePremium,Insurer,MortgageCode,Identifier,Phone,MobileNo,Mobile1
355,1,2C,HP-CV-FOTON-EV,29380,00102C00029380000002,DAMAR BAHADUR BIST,DHN/CV/F/24/25/00049,2025/03/16,2026/03/15,2026-03-15,5100000.0,75888.90,0.00,UNITED AJOD,C193327,SU PA PRA 02-001 JA 0497,9848780410,9848806061,9848806061
356,1,2C,HP-CV-FOTON-EV,29380,00102C00029380000002,DAMAR BAHADUR BIST,DHN/CVA/F/24/25/00014,2025/03/16,2026/03/15,2026-03-15,5100000.0,6359.30,0.00,UNITED AJOD,C9833275,SU PA PRA 02-001 JA 0497,9848780410,9848806061,9848806061
357,1,2C,HP-CV-FOTON-EV,29644,00102C00029644000003,GANESH ROKA NEPALI,KKD-MTR-APCVF-00004-81-82,2025/04/01,2026/03/31,2026-03-31,4680000.0,6894.92,6894.92,NECO(SELF),C9119372,LU PRA 03-001 KHA 0046,9858074345,9858074345,9858074345
358,1,2C,HP-CV-FOTON-EV,29644,00102C00029644000003,GANESH ROKA NEPALI,KKD-MTR-CVF-00130-81-82,2025/04/01,2026/03/31,2026-03-31,4680000.0,81238.75,81238.75,NECO(SELF),C1005601,LU PRA 03-001 KHA 0046,9858074345,9858074345,9858074345
359,1,2C,HP-CV-FOTON-EV,29676,00102C00029676000002,CHANDRA BAHADUR DANGI,NPJ/24/25/CV/F/00181,2025/04/01,2026/03/31,2026-03-31,4680000.0,72033.77,72033.77,PREMIER(SELF),C3768314,LU PRA 03-001 KHA 0047,9848163772,9848163772,9816532133


In [11]:
insurance_sum = insurance.groupby("MainCode")["InsurancePremium"].sum().reset_index()

In [12]:
print(len(insurance_sum))
print(insurance_sum["MainCode"].nunique())
print(insurance_sum["MainCode"].duplicated().sum())

830
830
0


In [13]:
duelist = duelist.drop_duplicates(subset="MainCode", keep = "last")
duelist_main = duelist_main.drop_duplicates(subset="MainCode", keep = "last")
insurance_sum = insurance_sum.drop_duplicates(subset="MainCode", keep = "last")

In [14]:
print(duelist['AgeingDays'].dtypes)

float64


In [15]:
duelist['AgeingDays'] = pd.to_numeric(duelist['AgeingDays'], errors='coerce')
duelist = duelist.sort_values(by='AgeingDays', ascending=False)
duelist.head(2)

,BBB,AT,AcTypeDesc,CC,OldClientCode,ClientCode,OldAcNum,MainCode,Name,Address,...,ModelDetails,LotNumber,DownPayment,VehicleModel,VehicleNo,Region,Obligor,DistrictCode,IntRateOfChgAc,ExpiryDateOfChgAc
17982,001,32,HP - HEAVY EQUIPMENT,01,NaN,00005629,NaN,00103200005629000002,AAYON NIRMAN SEWA PVT.LTD,"GHORAHI-10,DANG GHORAHI-10,DANG",...,JCB JS 220 TRACKED EXCAVATOR,NaN,40% Downpayment,7 HEAVY MACHINE,NaN,MID WESTERN,NaN,053,0.0,03/07/2025
18347,001,32,HP - HEAVY EQUIPMENT,01,001817,00003833,82001817041,00103200003833000004,RIDIPA CONSTRUCTION,Tewang-8 rolpa Rolpa District RAPTI ZONE,...,JCB JS205 LC Tracked Excavator,NaN,25% Downpayment,NaN,Ra 1 Ka 465,MID-WESTERN REGION,NaN,053,0.0,21/03/2025


In [16]:
pos = duelist.columns.get_loc("Name") + 1
duelist.insert(pos, "Ageing", np.nan)
duelist.insert(pos, "Loan Type", np.nan)
duelist.insert(pos, "OfficerName", np.nan)

pos = duelist.columns.get_loc("TotCharge") + 1
duelist.insert(pos, "OvDueWithInsurance", np.nan)
duelist.insert(pos, "InsurancePremium", np.nan)
duelist.insert(pos, "TotOvDue", np.nan)

pos = duelist.columns.get_loc("AgeingDays") + 1
duelist.insert(pos, "Bucket", np.nan)

pos = duelist.columns.get_loc("BranchName") + 1
duelist.insert(pos, "Dealer Name", np.nan)

In [17]:
# clean column names
duelist.columns = duelist.columns.str.strip()
duelist_main.columns = duelist_main.columns.str.strip()

# clean MainCode
duelist["MainCode"] = duelist["MainCode"].astype(str).str.strip()
duelist_main["MainCode"] = duelist_main["MainCode"].astype(str).str.strip()

cols = ["OfficerName", "Loan Type", "Dealer Name"]

df = duelist.merge(
    duelist_main[["MainCode"] + cols],
    on="MainCode",
    how="left",
    suffixes=("", "_ref")
)

for col in cols:
    df[col] = df[col].astype(str).str.strip()
    df[col] = df[col].replace(r'^(|None|nan|\s+)$', pd.NA, regex=True)
    df[col] = df[col].fillna(df[f"{col}_ref"])

df.drop(columns=[c + "_ref" for c in cols], inplace=True)

In [18]:
print(len(df))
df.head(2)

26767


,BBB,AT,AcTypeDesc,CC,OldClientCode,ClientCode,OldAcNum,MainCode,Name,OfficerName,...,ModelDetails,LotNumber,DownPayment,VehicleModel,VehicleNo,Region,Obligor,DistrictCode,IntRateOfChgAc,ExpiryDateOfChgAc
0,001,32,HP - HEAVY EQUIPMENT,01,NaN,00005629,NaN,00103200005629000002,AAYON NIRMAN SEWA PVT.LTD,Dang-Chronic Account-Mukesh Dahal-JCB,...,JCB JS 220 TRACKED EXCAVATOR,NaN,40% Downpayment,7 HEAVY MACHINE,NaN,MID WESTERN,NaN,053,0.0,03/07/2025
1,001,32,HP - HEAVY EQUIPMENT,01,001817,00003833,82001817041,00103200003833000004,RIDIPA CONSTRUCTION,Kathmandu-Chronic Account-Mukesh Dahal-JCB,...,JCB JS205 LC Tracked Excavator,NaN,25% Downpayment,NaN,Ra 1 Ka 465,MID-WESTERN REGION,NaN,053,0.0,21/03/2025


In [19]:
# insurance = insurance.rename(columns={
#     "prem":"InsurancePremium"
# })

# merge
df = df.merge(
    insurance_sum[["MainCode", "InsurancePremium"]],
    on="MainCode",
    how="left",
    suffixes=("", "_ref")
)
# fill missing values
df["InsurancePremium"] = df["InsurancePremium"].fillna(df["InsurancePremium_ref"])

# remove extra columns
df.drop(columns=["InsurancePremium_ref"], inplace=True)

# fill remaining empty with 0
df["InsurancePremium"] = df["InsurancePremium"].fillna(0)

In [20]:
print(len(df))
print(df["MainCode"].nunique())
print(df["MainCode"].duplicated().sum())
df.head(2)

26767
26767
0


,BBB,AT,AcTypeDesc,CC,OldClientCode,ClientCode,OldAcNum,MainCode,Name,OfficerName,...,ModelDetails,LotNumber,DownPayment,VehicleModel,VehicleNo,Region,Obligor,DistrictCode,IntRateOfChgAc,ExpiryDateOfChgAc
0,001,32,HP - HEAVY EQUIPMENT,01,NaN,00005629,NaN,00103200005629000002,AAYON NIRMAN SEWA PVT.LTD,Dang-Chronic Account-Mukesh Dahal-JCB,...,JCB JS 220 TRACKED EXCAVATOR,NaN,40% Downpayment,7 HEAVY MACHINE,NaN,MID WESTERN,NaN,053,0.0,03/07/2025
1,001,32,HP - HEAVY EQUIPMENT,01,001817,00003833,82001817041,00103200003833000004,RIDIPA CONSTRUCTION,Kathmandu-Chronic Account-Mukesh Dahal-JCB,...,JCB JS205 LC Tracked Excavator,NaN,25% Downpayment,NaN,Ra 1 Ka 465,MID-WESTERN REGION,NaN,053,0.0,21/03/2025


In [21]:
age = df['AgeingDays']

conditions1 = [
    age.isna() | (age == 0),
    age <= 30,
    age <= 60,
    age <= 90,
    age <= 120,
    age <= 180,
    age <= 365,
    age > 365
]

choices1 = [
    "Regular",
    "1-30 Days",
    "31-60 Days",
    "61-90 Days",
    "91-120 Days",
    "121-180 Days",
    "181-365 Days",
    "Above 365 Days"
]

df['Ageing'] = np.select(conditions1, choices1, default="Unknown")

conditions2 = [
    age.isna() | (age == 0),
    age <= 90,
    age <= 180,
    age <= 365,
    age > 365
]

choices2 = [
    "Regular",
    "1-90 Days",
    "91-180 Days",
    "181-365 Days",
    "Above 365 Days"
]

df['Bucket'] = np.select(conditions2, choices2, default="Unknown")

In [22]:
cols = ['OutstandingBaln','IntDrAmt','PenalIntAmt','IntOnInt',
        'OvDuePrin','PastDuedInt','TotCharge']
# df[cols].dtypes
df[cols] = df[cols].apply(pd.to_numeric, errors='coerce')
df[cols].dtypes

OutstandingBaln    float64
IntDrAmt           float64
PenalIntAmt        float64
IntOnInt           float64
OvDuePrin          float64
PastDuedInt        float64
TotCharge          float64
dtype: object

In [23]:
df['TotOvDue'] = np.where(
    df['Remarks'] == "Expired",
    -df['OutstandingBaln'] + df['IntDrAmt'] + df['TotCharge'],
    df['PenalIntAmt'] + df['IntOnInt'] + df['OvDuePrin'] + df['PastDuedInt'] + df['TotCharge']
)

df['OvDueWithInsurance'] = df['TotOvDue'] + df['InsurancePremium']

In [24]:
keywords = ["sold out", "court case"]

df.loc[
    df["OfficerName"].str.contains("|".join(keywords), case=False, na=False),
    "Bucket"
] = "Above 365 Days"

In [25]:
# df.to_excel(output_file, index=False, sheet_name="Mainsheet")

In [26]:
cols = ["OfficerName", "Loan Type", "Dealer Name"]

# Prepare reference table: one row per combination
df_ref = duelist_main[["AcTypeDesc", "BranchName"] + cols].drop_duplicates(
    subset=["AcTypeDesc", "BranchName"], keep="last"
)

# Merge with df
final_df = df.merge(
    df_ref,
    on=["AcTypeDesc", "BranchName"],
    how="left",
    suffixes=("", "_ref")
)

# Fill missing values
for col in cols:
    final_df[col] = final_df[col].astype(str).str.strip().replace(r'^(|None|nan|\s+)$', pd.NA, regex=True)
    final_df[col] = final_df[col].fillna(final_df[f"{col}_ref"])

# Drop temporary reference columns
final_df.drop(columns=[f"{col}_ref" for col in cols], inplace=True)

In [27]:
# Merge only on BranchName for remaining missing values ---
df_ref_branch = duelist_main[["BranchName"] + cols].drop_duplicates(subset=["BranchName"], keep="last")

final_df = final_df.merge(
    df_ref_branch,
    on="BranchName",
    how="left",
    suffixes=("", "_branch")
)

# Fill missing values from BranchName merge
for col in cols:
    final_df[col] = final_df[col].fillna(final_df[f"{col}_branch"])

# Drop temporary branch reference columns
final_df.drop(columns=[f"{col}_branch" for col in cols], inplace=True)

In [28]:
final_df.to_excel(output_file, index=False, sheet_name="Mainsheet")